In [ ]:
import numpy as np
import mpmath as mp
from scipy.optimize import minimize
import cvxpy as cp
import csv
import pandas as pd
import matplotlib.pyplot as plt

# Certified grid verification of the MHR lower bound

For each capacity C, the code constructs conservative endpoints for the interior d-interval, divides that interval into exactly N equal subintervals, and makes one primary CLARABEL solve attempt at the rounded midpoint of every subinterval. Additional ECOS solves may be used only to repair conspicuous numerical outliers.

Paper configurations
--------------------
- Table 5: C=2,...,100, N=12000, target=0.9906, solver=CLARABEL, and coefficient_digits=6.
- Figure 2 data: C=101,...,974, N=100, with the same target, solver, and coefficient precision.
- The remaining defaults are midpoint_digits=5, endpoint_digits=10, bound_digits=10, and mp.mp.dps=80.

Main numerical conventions
--------------------------
1. Every midpoint is rounded to ``midpoint_digits`` decimal places before it is sent to CVXPY.

2. The solver values of b and s_1,...,s_C are rounded to ``coefficient_digits`` decimal places.

3. The solver value of a is not used. After b and s_1,...,s_C are fixed,
   a is recomputed from all dual constraints, rounded downward, and moved
   one additional decimal unit into the feasible region. All C+1 slacks
   are then recomputed; a line is accepted only if every slack is positive.

4. A dual-feasible line a+bd is valid for every d. The midpoint is used
   only to obtain a numerically strong line for its corresponding interval.

Solver failures and neighboring-line fallback
----------------------------------------------
A solver failure is treated as a numerical failure, not as a failed mathematical constraint. No unverified solver output is ever used.

- If failures occur before the first successful midpoint, those intervals are evaluated with the first subsequently verified line (next_verified_line).
- After the first success, a failed midpoint temporarily uses the most recent verified line (previous_verified_line), and the solver object is restarted before the next midpoint.
- After all primary attempts, every interval is checked again with the nearest verified line on its left and on its right. The code keeps whichever already verified line gives the largest certified lower bound on that interval.
- If CLARABEL produces no verified line anywhere, the computation raises an error and reports no certificate.

This fallback is rigorous because every retained line has already passed the global dual-feasibility check and is valid for every value of d. Using a line obtained at a neighboring midpoint can make the bound less sharp, but it cannot make the reported lower bound invalid.

Outlier repair
--------------
After the neighboring-line comparison, an interval is flagged as a numerical outlier when its certified bound is more than 0.001 below the largest bound among the two intervals on either side (outlier_radius=2, truncated at the endpoints). A fresh ECOS solve is attempted at that interval's midpoint. The ECOS line replaces the existing line only if it independently passes the same rounding and feasibility checks and gives a strictly larger certified bound. If ECOS fails or does not improve the bound, the previous certified line is retained.

Reading the output
------------------
For each capacity, passes_target=True means that the minimum certified interior-interval bound is at least 0.9906. The two analytical tails are already covered at the target value, so global_bound is the minimum of 0.9906 and the interior bound. The CSV records the worst interval, its rounded coefficients' minimum dual slack, the source of the line used there, and any solver message.


In [ ]:
# Numerical precision + Erlang loss formula + [ d_low, d_high  ]
# ------------------------------------------------------------

# High precision is used when evaluating exponentials, repairing a,
# checking the dual constraints, and computing the interval bounds.
mp.mp.dps = 80

# ------------------------------------------------------------
# Decimal-conversion and rounding helpers
# ------------------------------------------------------------

def mpf(x):
    """
    Convert x to a high-precision mp.mpf.
    """
    if isinstance(x, mp.mpf):
        return x
    if isinstance(x, (float, np.floating)):
        return mp.mpf(repr(float(x)))
    return mp.mpf(str(x))


def unit(digits):
    """Return one unit in the last decimal place, namely 10^{-digits}."""
    return mp.power(10, -digits)


def round_nearest(x, digits):
    """Round x to the nearest number with the specified decimal digits."""
    scale = mp.power(10, digits)
    return mp.nint(mpf(x) * scale) / scale


def round_down(x, digits):
    """Round x downward to the specified number of decimal digits."""
    scale = mp.power(10, digits)
    return mp.floor(mpf(x) * scale) / scale


def round_up(x, digits):
    """Round x upward to the specified number of decimal digits."""
    scale = mp.power(10, digits)
    return mp.ceil(mpf(x) * scale) / scale


# ------------------------------------------------------------
# Erlang loss functions + Carried load
# ------------------------------------------------------------

def service_level(C, x):
    """
    Compute 1-B_C(x), where B_C is the Erlang blocking probability.

    The recursion  B_i(x) = x B_{i-1}(x) / (i+x B_{i-1}(x)),  B_0(x)=1,    avoids directly evaluating factorials and large powers.
    """
    x = mpf(x)
    B = mp.mpf(1)

    for i in range(1, C + 1):
        B = x * B / (i + x * B)

    return 1 - B


def A(C, x):
    """Compute A_C(x)=x(1-B_C(x))."""
    x = mpf(x)
    return x * service_level(C, x)


# ------------------------------------------------------------
# Conservative endpoints of the interior d-interval
# ------------------------------------------------------------

def conservative_d_bounds(
    C,
    target=0.97,
    endpoint_digits=10,
    bisection_steps=160
):
    """
    Construct conservative endpoints [d_lower,d_upper].

    The left tail is handled by    1-B_C(d) >= target.

    Bisection finds the largest numerical point satisfying this inequality.
    The returned d_lower is then rounded downward and moved one additional decimal unit to the left.

    The right tail is handled by    C / (C^2-(C-1)d) >= target.

    Solving the equality gives d = (C^2-C/target)/(C-1).

    The returned d_upper is rounded upward and moved one additional decimal unit to the right.

    Therefore, only [d_lower,d_upper] needs the numerical dual calculation.
    """
    target = mpf(target)
    left = mp.mpf(0)
    right = mp.mpf(C)

    # Maintain service_level(C,left) >= target and move left upward.
    for _ in range(bisection_steps):
        midpoint = (left + right) / 2

        if service_level(C, midpoint) >= target:
            left = midpoint
        else:
            right = midpoint

    d_lower = max(
        mp.mpf(0),
        round_down(left, endpoint_digits) - unit(endpoint_digits)
    )

    exact_upper = (C * C - C / target) / (C - 1)
    d_upper = round_up(exact_upper, endpoint_digits) + unit(endpoint_digits)

    return d_lower, d_upper

In [ ]:
# 2. CVXPY formulation of the dual problem
# ------------------------------------------------------------

class DualOracle:
    """
    Numerically solve the dual problem
        maximize    a+b d
        subject to  a <= -exp(s_1-1),
                    a+i b <= i s_i-exp(s_{i+1}-1), i=1,...,C-1,
                    a+C b <= C s_C.
                    
    The CVXPY problem is constructed only once for a given C. The parameter
    d is changed when the code moves from one grid interval to the next.

    The solver output is only a candidate. The rounded coefficients are
    repaired and independently checked by make_feasible_line.
    """

    def __init__(self, C, solver="ECOS"):
        self.C = C
        self.solver = solver

        self.d = cp.Parameter(nonneg=True)
        self.s = cp.Variable(C)
        self.a = cp.Variable()
        self.b = cp.Variable()

        constraints = [
            self.a + cp.exp(self.s[0] - 1) <= 0
        ]

        for i in range(1, C):
            constraints.append(
                self.a + i * self.b - i * self.s[i - 1] + cp.exp(self.s[i] - 1)  <= 0
            )

        constraints.append(
            self.a + C * self.b - C * self.s[C - 1] <= 0
        )

        self.problem = cp.Problem(
            cp.Maximize(self.a + self.b * self.d),
            constraints
        )

    def solve(self, d_mid):
        """
        Solve the dual numerically at d_mid.

        The returned s_i and b are subsequently rounded. The returned solver
        objective is retained only for diagnostics and is not used as a
        certified lower bound.
        """
        self.d.value = float(d_mid)

        self.problem.solve(
            solver=self.solver,
            warm_start=True,
            verbose=False
        )

        acceptable_statuses = {
            cp.OPTIMAL,
            cp.OPTIMAL_INACCURATE
        }

        if self.problem.status not in acceptable_statuses:
            raise RuntimeError(
                f"Dual solve failed at d={d_mid}. "
                f"Status: {self.problem.status}"
            )

        s = np.asarray(self.s.value, dtype=float).reshape(-1)
        b = float(self.b.value)

        if not np.all(np.isfinite(s)):
            raise RuntimeError("The solver returned nonfinite s values.")

        if not np.isfinite(b):
            raise RuntimeError("The solver returned a nonfinite b value.")

        return {
            "s": s,
            "b": b,
            "solver_value": float(self.problem.value)
        }


# ------------------------------------------------------------
# Construction and verification of a rounded feasible dual line
# ------------------------------------------------------------

def make_feasible_line(C, candidate, coefficient_digits=6):
    """
    Convert the numerical solver output into rounded dual-feasible values.

    First round b and s_1,...,s_C to ``coefficient_digits`` decimal places.
    After these values are fixed, every dual constraint gives an upper bound on a:

        a <= -exp(s_1-1),
        a <= i s_i-exp(s_{i+1}-1)-i b, i=1,...,C-1,
        a <= C s_C-C b.

    We take the smallest of these upper bounds, round it downward, and then
    subtract one additional decimal unit. Hence a is not copied from CVXPY.

    Finally, all C+1 dual slacks are recomputed directly. If any slack is
    nonpositive, the code stops rather than using an infeasible line.

    Once feasible, weak duality gives

        I_C(d) >= a+b d

    for every d, not only for the midpoint at which the candidate was found.
    """
    s = [
        round_nearest(value, coefficient_digits)
        for value in candidate["s"]
    ]
    b = round_nearest(candidate["b"], coefficient_digits)

    # Each element of allowed_a is the largest a allowed by one constraint.
    allowed_a = [
        -mp.exp(s[0] - 1)
    ]

    for i in range(1, C):
        allowed_a.append( i * s[i - 1] - mp.exp(s[i] - 1) - i * b
        )

    allowed_a.append(
        C * s[C - 1] - C * b
    )

    # Round a downward and move one more unit into the feasible region.
    a = (
        round_down(min(allowed_a), coefficient_digits)
        - unit(coefficient_digits)
    )

    # Recompute all dual slacks using the rounded values shown in the output.
    slacks = [
        -mp.exp(s[0] - 1) - a
    ]

    for i in range(1, C):
        slacks.append(  i * s[i - 1]  - mp.exp(s[i] - 1)  - a - i * b
        )

    slacks.append(
        C * s[C - 1] - a - C * b
    )

    minimum_slack = min(slacks)

    if minimum_slack <= 0:
        raise RuntimeError(
            "Rounded dual solution is infeasible. "
            f"Minimum slack: {minimum_slack}"
        )

    return {
        "s": s,
        "a": a,
        "b": b,
        "minimum_dual_slack": minimum_slack,
        "solver_value": candidate["solver_value"]
    }


# ------------------------------------------------------------
# Lower bound on one fixed grid interval
# ------------------------------------------------------------

def bound_one_interval(
    C,
    d_left,
    d_right,
    line,
    bound_digits=10
):
    """
    Lower-bound the objective over d in [d_left,d_right].

    Since the rounded line is dual feasible,

        I_C(d)/d >= b+a/d.

    Its minimum is therefore attained at one of the two endpoints:

        I_C(d)/d >= min{b+a/d_left, b+a/d_right}.

    Since A_C is increasing and d <= d_right,

        A_C(exp(I_C(d)/d))/d >= A_C(exp(log_omega_lower))/d_right.

    The final value is rounded downward to ``bound_digits`` decimal places.
    An additional smaller margin is removed before rounding to reduce the
    effect of the last high-precision numerical evaluation.
    """
    a = line["a"]
    b = line["b"]

    log_omega_lower = min(
        b + a / d_left,
        b + a / d_right
    )

    raw_bound = A(C, mp.exp(log_omega_lower)) / d_right

    return max(
        mp.mpf(0),
        round_down(
            raw_bound - unit(bound_digits + 10),
            bound_digits
        )
    )


# ------------------------------------------------------------
# Uniform, nonadaptive subdivision
# ------------------------------------------------------------

def uniform_certified_bound(
    C,
    N,
    target=0.97,
    solver="ECOS",
    midpoint_digits=5,
    coefficient_digits=6,
    endpoint_digits=10,
    bound_digits=10
):
    """
    Verify a lower bound using exactly N equal subintervals.

    Procedure
    ---------
    1. Compute the conservative interior interval [d_lower,d_upper].

    2. Divide it into N equal intervals. 

    3. For each interval, round its midpoint to ``midpoint_digits`` decimal
       places and solve the dual at that rounded midpoint.

    4. Round b and s_1,...,s_C, recompute a downward, and verify all dual
       constraints.

    5. Use this globally valid affine line to lower-bound the objective on
       the entire interval.

    6. Return the smallest interval bound as the interior bound.

    The two analytical tails are established only at the requested target,
    so the reported global bound is min(target,interior_bound).

    Important
    ---------
    A failure to reach the target may only mean that N is too small.   
    It simply reports the worst interval and its rounded dual coefficients.
    """
    if C < 2:
        raise ValueError("C must be at least 2.")

    if N <= 0:
        raise ValueError("N must be positive.")

    target = mpf(target)

    d_lower, d_upper = conservative_d_bounds(
        C,
        target=target,
        endpoint_digits=endpoint_digits
    )

    # The analytical endpoint arguments already cover the whole domain.
    if d_lower >= d_upper:
        return {
            "C": C,
            "N": N,
            "d_lower": d_lower,
            "d_upper": d_upper,
            "interior_bound": mp.inf,
            "global_bound": target,
            "passes_target": True,
            "worst_interval": None,
            "intervals": []
        }

    oracle = DualOracle(C, solver=solver)
    width = (d_upper - d_lower) / N
    intervals = []

    def record_interval(j, d_left,  d_right, d_mid, line, solution_source, solver_message):

        interval_bound = bound_one_interval(C, d_left, d_right,line, bound_digits=bound_digits)

        intervals.append({
            "index": j,
            "d_left": d_left,
            "d_right": d_right,
            "midpoint": d_mid,
            "bound": interval_bound,
            "a": line["a"],
            "b": line["b"],
            "s": line["s"],
            "minimum_dual_slack": line["minimum_dual_slack"],
            "solution_source": solution_source,
            "solver_message": solver_message
        })

    # --------------------------------------------------------
    # Find the first successful verified CLARABEL line.
    # Each midpoint is solved at most once.
    # --------------------------------------------------------

    line = None
    initial_failures = {}
    last_error = ""

    for seed_index in range(N):
        d_left = d_lower + seed_index * width
        d_right = d_lower + (seed_index + 1) * width

        d_mid = round_nearest( (d_left + d_right) / 2, midpoint_digits)

        try:
            candidate = oracle.solve(d_mid)

            new_line = make_feasible_line(
                C,
                candidate,
                coefficient_digits=coefficient_digits
            )

        except (cp.error.SolverError, RuntimeError) as error:
            last_error = str(error)
            initial_failures[seed_index] = last_error

            # Use a fresh solver instance for the next midpoint.
            oracle = DualOracle(C, solver=solver)

        else:
            line = new_line
            break

    if line is None:
        raise RuntimeError(
            "CLARABEL did not produce any verified feasible line. "
            f"Last error: {last_error}"
        )

    # --------------------------------------------------------
    # Backfill the initial failed intervals using the first
    # successful globally feasible line.
    # --------------------------------------------------------

    for j in range(seed_index):
        d_left = d_lower + j * width
        d_right = d_lower + (j + 1) * width
        d_mid = round_nearest( (d_left + d_right) / 2, midpoint_digits )
        record_interval( j, d_left, d_right, d_mid, line, "next_verified_line", initial_failures[j])

    # Record the interval where the first successful line was found.
    d_left = d_lower + seed_index * width
    d_right = d_lower + (seed_index + 1) * width
    d_mid = round_nearest((d_left + d_right) / 2, midpoint_digits)
    record_interval(seed_index, d_left, d_right, d_mid, line, "current_midpoint", "")

    # --------------------------------------------------------
    # Continue after the first successful midpoint.
    # Later failures reuse the preceding verified line.
    # --------------------------------------------------------

    for j in range(seed_index + 1, N):
        d_left = d_lower + j * width
        d_right = d_lower + (j + 1) * width

        d_mid = round_nearest((d_left + d_right) / 2, midpoint_digits)

        try:
            candidate = oracle.solve(d_mid)

            new_line = make_feasible_line(
                C,
                candidate,
                coefficient_digits=coefficient_digits
            )

        except (cp.error.SolverError, RuntimeError) as error:
            solution_source = "previous_verified_line"
            solver_message = str(error)

            # Discard the failed solver state, but retain `line`.
            oracle = DualOracle(C, solver=solver)

        else:
            line = new_line
            solution_source = "current_midpoint"
            solver_message = ""

        record_interval(
            j,
            d_left,
            d_right,
            d_mid,
            line,
            solution_source,
            solver_message
        )

    # --------------------------------------------------------
    # Re-evaluate every interval using the nearest verified solver lines on both sides.
    # --------------------------------------------------------

    interval_snapshot = [interval.copy()  for interval in intervals ]

    number_of_intervals = len(interval_snapshot)

    # nearest_left[j] is the closest interval strictly to the left whose solver call succeeded at its own midpoint.
    nearest_left = [None] * number_of_intervals
    last_successful_index = None
    for j in range(number_of_intervals):
        nearest_left[j] = last_successful_index
        if (interval_snapshot[j]["solution_source"] == "current_midpoint" ):
            last_successful_index = j

    # nearest_right[j] is the closest interval strictly to the right whose solver call succeeded at its own midpoint.
    nearest_right = [None] * number_of_intervals
    last_successful_index = None
    for j in range(number_of_intervals - 1, -1, -1):
        nearest_right[j] = last_successful_index
        if (interval_snapshot[j]["solution_source"]== "current_midpoint" ):
            last_successful_index = j


    # For every interval, compare its currently stored line with
    # the nearest successful lines on the left and right.
    for j in range(number_of_intervals):
        current_interval = interval_snapshot[j]

        best_bound = current_interval["bound"]
        best_line_index = j

        nearby_indices = []

        if nearest_left[j] is not None:
            nearby_indices.append(nearest_left[j])

        if nearest_right[j] is not None:
            nearby_indices.append(nearest_right[j])

        for nearby_index in nearby_indices:
            nearby_interval = interval_snapshot[nearby_index]

            nearby_line = {
                "a": nearby_interval["a"],
                "b": nearby_interval["b"],
                "s": nearby_interval["s"],
                "minimum_dual_slack": nearby_interval[  "minimum_dual_slack"],
            }

            nearby_bound = bound_one_interval(
                C,
                current_interval["d_left"],
                current_interval["d_right"],
                nearby_line,
                bound_digits=bound_digits,
            )

            if nearby_bound > best_bound:
                best_bound = nearby_bound
                best_line_index = nearby_index

        # Replace only when a nearby line gives a strictly better
        # certified bound on the current interval.
        if best_line_index != j:
            best_line_interval = interval_snapshot[
                best_line_index
            ]

            intervals[j]["bound"] = best_bound
            intervals[j]["a"] = best_line_interval["a"]
            intervals[j]["b"] = best_line_interval["b"]
            intervals[j]["s"] = best_line_interval["s"]
            intervals[j]["minimum_dual_slack"] = (best_line_interval["minimum_dual_slack"])
            intervals[j]["solver_message"] = ""
            intervals[j]["line_midpoint"] = (best_line_interval["midpoint"] )

            if best_line_index < j:
                intervals[j]["solution_source"] = ( "previous_verified_line"  )
            else:
                intervals[j]["solution_source"] = ( "next_verified_line")

    # --------------------------------------------------------
    # Retry conspicuous local outliers with a fresh ECOS solver.
    # --------------------------------------------------------

    outlier_gap = mpf("0.001")
    outlier_radius = 2

    # Detect outliers using a fixed snapshot so that repairing one interval does not change the detection of another interval.
    outlier_snapshot = [ interval.copy() for interval in intervals]

    for j in range(len(outlier_snapshot)):
        current_interval = outlier_snapshot[j]

        nearby_indices = [k for k in range( max(0, j - outlier_radius), min(len(outlier_snapshot), j + outlier_radius + 1), ) if k != j]

        if not nearby_indices:
            continue

        nearby_reference = max(outlier_snapshot[k]["bound"] for k in nearby_indices)

        # Retry only when the current interval is substantially lower than at least one nearby interval.
        if ( nearby_reference - current_interval["bound"] <= outlier_gap):
            continue

        try:
            # Use a fresh ECOS instance for this midpoint.
            ecos_oracle = DualOracle(C, solver="ECOS",)
            ecos_candidate = ecos_oracle.solve(current_interval["midpoint"])
            ecos_line = make_feasible_line(
                C,
                ecos_candidate,
                coefficient_digits=coefficient_digits,
            )

            ecos_bound = bound_one_interval(
                C,
                current_interval["d_left"],
                current_interval["d_right"],
                ecos_line,
                bound_digits=bound_digits,
            )

        except (cp.error.SolverError, RuntimeError) as error:
            previous_message = intervals[j]["solver_message"]

            if previous_message:
                intervals[j]["solver_message"] = ( previous_message + f" | ECOS retry failed: {error}")
            else:
                intervals[j]["solver_message"] = ( f"ECOS retry failed: {error}" )

        else:
            # Replace the existing line only when ECOS gives a
            # strictly better certified bound.
            if ecos_bound > intervals[j]["bound"]:
                old_bound = intervals[j]["bound"]

                intervals[j]["bound"] = ecos_bound
                intervals[j]["a"] = ecos_line["a"]
                intervals[j]["b"] = ecos_line["b"]
                intervals[j]["s"] = ecos_line["s"]
                intervals[j]["minimum_dual_slack"] = ( ecos_line["minimum_dual_slack"]    )
                intervals[j]["solution_source"] = (  "ECOS_outlier_retry")
                intervals[j]["solver_message"] = (
                    f"ECOS improved outlier from "
                    f"{old_bound} to {ecos_bound}."
                )
                intervals[j]["line_midpoint"] = (current_interval["midpoint"])

    worst_interval = min(
        intervals,
        key=lambda interval: interval["bound"]
    )
    interior_bound = worst_interval["bound"]

    return {
        "C": C,
        "N": N,
        "d_lower": d_lower,
        "d_upper": d_upper,
        "interior_bound": interior_bound,
        "global_bound": min(target, interior_bound),
        "passes_target": interior_bound >= target,
        "worst_interval": worst_interval,
        "intervals": intervals
    }




In [ ]:
# Run a range of capacities and save a compact CSV summary
# ------------------------------------------------------------

def run_capacity_range(
    C_start,
    C_end,
    N,
    target=0.97,
    solver="CLARABEL",
    coefficient_digits=6,
    filename="uniform_summary.csv"
):
    """
    Run uniform_certified_bound for C=C_start,...,C_end.

    The CSV records only a compact summary for each C. In particular, it
    records the worst interval and its minimum dual slack, rather than all N
    rounded dual solutions. The full interval information remains available
    in the result returned by uniform_certified_bound when one C is run.
    """
    if C_start > C_end:
        raise ValueError("C_start must not exceed C_end.")

    rows = []

    for C in range(C_start, C_end + 1):
        result = uniform_certified_bound(
            C=C,
            N=N,
            target=target,
            solver=solver,
            coefficient_digits=coefficient_digits
        )
        worst = result["worst_interval"]

        rows.append({
            "C": C,
            "N": N,
            "global_bound": mp.nstr(result["global_bound"], 20),
            "interior_bound": mp.nstr(result["interior_bound"], 20),
            "d_lower": mp.nstr(result["d_lower"] ,20),
            "d_upper": mp.nstr(result["d_upper"] ,20),
            "passes_target": result["passes_target"],
            "worst_left": "" if worst is None else mp.nstr(worst["d_left"], 20),
            "worst_right": "" if worst is None else mp.nstr(worst["d_right"], 20),
            "worst_midpoint": "" if worst is None else mp.nstr(worst["midpoint"], 20),
            "minimum_dual_slack": "" if worst is None else mp.nstr(worst["minimum_dual_slack"], 20),
            "solution_source": (
                "" if worst is None else worst["solution_source"]
            ),
            "solver_message": (
                "" if worst is None else worst["solver_message"]
            )
        })

        print(
            f"C={C}, bound={result['global_bound']}, "
            f"passes={result['passes_target']}"
        )

    with open(filename, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    return rows



# Result 2 - 100

In [ ]:
run_capacity_range(
    C_start=2,
    C_end=100,
    N=12000,
    target=0.9906,
    solver="CLARABEL",
    coefficient_digits =6,
    filename="2To100.csv"
)    

# Result 101 - 974


In [ ]:
run_capacity_range(
    C_start=101,
    C_end=974,
    N=100,
    target=0.9906,
    solver="CLARABEL",
    coefficient_digits =6,
    filename="101To974.csv"
)    

In [ ]:
df = pd.read_csv("101To974.csv")
df = df[df["C"].between(101, 974)].sort_values("C")

# Numerically stable computation of
# 1 - ((C-1)^C / C!) / sum_{i=0}^C ((C-1)^i / i!)
def comparison_curve(C):
    ratio = 1.0
    normalized_sum = 1.0

    for i in range(C, 0, -1):
        ratio *= i / (C - 1)
        normalized_sum += ratio

    return 1.0 - 1.0 / normalized_sum

df["comparison_curve"] = df["C"].apply(comparison_curve)

# plt.figure(figsize=(12, 6))
plt.figure(figsize=(9.5,4),dpi=200)

plt.plot( df["C"], df["global_bound"], label="MHR Guarantee", color='green')
# plt.plot(df["C"],df["comparison_curve"],label=r"$1-\frac{(C-1)^C/C!}{\sum_{i=0}^C(C-1)^i/i!}$",linewidth=1.5, linestyle="--",)
plt.axhline( y=0.9755, color="darkgray",linestyle=":",linewidth=1.5,)

plt.xlabel(r"$C$")
plt.ylabel('Guarantee')
plt.xlim(101, 974)
plt.xticks([101,200,300,400,500,600,700,800,900,974])
plt.yticks([0.97, 0.9755,0.98, 0.985, 0.99])
plt.legend()
plt.tight_layout()

# Optional:
# plt.savefig("guarantee_101_to_974.pdf", bbox_inches="tight")
# plt.savefig("guarantee_101_to_974.png", dpi=300, bbox_inches="tight")

plt.savefig('MHR101',dpi=200,bbox_inches='tight') 